# Database Management Systems: Week 12 - In-Depth Notes

## Week 12 Overview: Query Optimization, Scalability, NoSQL, and Course Conclusion

Week 12 is the final week of the Database Management Systems course. It covers the remaining pillars of modern data management:

1. **Query Processing and Optimization** (Modules 56-57): How SQL queries are translated into relational algebra, how operations are executed, and how the optimizer generates equivalent plans and chooses the cheapest one.

2. **RDBMS Performance and Architecture** (Module 58): How the physical architecture—centralized, client-server, parallel, or distributed—affects throughput, response time, availability, and scalability.

3. **Non-Relational DBMS: NoSQL** (Module 59): The emergence of non-relational systems for big data, the CAP theorem, BASE properties, and the four major NoSQL data models.

4. **Widely Used DBMSs and Course Summarization** (Module 60): A comparative overview of leading RDBMSs, market trends, and a full recap of the course.

These topics complete the journey from conceptual design to large-scale, production-grade data systems.

---

## Module 56: Query Processing and Optimization – Part 1: Overview and Evaluation Algorithms

### 56.1. Query Processing Pipeline

When a user submits an SQL query, the DBMS processes it through several stages:

```
SQL Query
   ↓
Parser and Translator → Relational Algebra Expression
   ↓
Optimizer → Execution Plan (annotated relational algebra tree)
   ↓
Evaluation Engine → Query Output
```

- **Parser**: Checks SQL syntax, translates into a parse tree.
- **Translator**: Converts the parse tree into a relational algebra expression.
- **Optimizer**: Generates equivalent relational algebra expressions, estimates costs, chooses the best **execution plan**.
- **Evaluation Engine**: Executes the plan step-by-step against stored data, using indexes, buffers, etc.

The optimizer uses **statistical information** from the system catalog (number of tuples, block counts, index heights, etc.) to estimate costs.

### 56.2. Example of Query Processing

**SQL Query:**
```sql
SELECT salary
FROM instructor
WHERE salary < 75000;
```

**Possible relational algebra expressions:**

- **Option 1:** `π_salary(σ_{salary < 75000}(instructor))`
- **Option 2:** `σ_{salary < 75000}(π_salary(instructor))`

Both are equivalent, but which is better? The first may be better because it selects on the full instructor relation using an index on salary; the second first projects salary to reduce width but may not be allowed if there is no index. The optimizer considers such trade-offs.

### 56.3. Measures of Query Cost

Cost is primarily measured by **total elapsed time** for answering a query. Major components:

1. **Disk Access**: Reading/writing blocks between disk and main memory.
2. **CPU Time**: Processing data in memory.
3. **Network Communication**: For distributed systems (ignored in this module for simplicity).

We focus on disk access because it dominates.

**Disk access components:**
- **Number of seeks**: Positioning the read/write head to the correct track.
- **Number of block transfers**: Reading/writing a block from/to disk.

**Cost Formula:**
```
Cost = b * tT + S * tS
```
where:
- `b` = number of block transfers
- `tT` = time to transfer one block
- `S` = number of seeks
- `tS` = time for one seek

We often ignore CPU cost and use **worst-case estimates**.

### 56.4. Selection Algorithms

Given a relation with `b_r` blocks and `n_r` tuples, we evaluate selection `σ_θ(r)`.

**A1: Linear Search (file scan)**
- Scan all blocks sequentially.
- Cost: `b_r` block transfers + 1 seek (initial seek).

**A2: Linear Search, Equality on Key**
- Since key is unique, stop when match found.
- Average case: half the blocks scanned.
- Cost: average `b_r / 2` transfers; worst case `b_r`.

**A3: Primary Index, Equality on Key (B+ tree)**
- Traverse index from root to leaf: `h_i + 1` accesses (height of index + 1 for data block).
- Each access involves seek + transfer.
- Cost: `(h_i + 1) * (t_T + t_S)`.

**A4: Primary Index, Equality on Non-Key**
- Multiple records may match.
- Cost: `h_i` index accesses + `b` data block transfers (where `b` blocks contain matching records).

**A5: Secondary Index, Equality**
- Similar to A3/A4 depending on key/non-key.
- Cost: `h_i + n` for non-key if each matching record is in a different block.

**A6: Comparison Conditions (range queries)**
- With primary index: similar to A4.
- With secondary index: similar to A4, may require more seeks.

**Complex selections (AND/OR):**
- **Conjunction (AND):** Use index on one condition, fetch tuples, test remaining conditions in memory. Or use intersection of identifiers from multiple indexes.
- **Disjunction (OR):** Use union of identifiers from available indexes; if no index, linear scan.
- **Negation (NOT):** Linear scan typically.

### 56.5. Sorting

When a query requires sorted output or duplicate elimination, we may need sorting.

**Options:**
1. **Index scan**: If a B+ tree index exists on the sort attribute, traverse leaf nodes in order. But this may cause random disk accesses for each tuple (bad if records are scattered).
2. **In-memory sort**: If data fits in memory, use quicksort.
3. **External sort-merge**: If data is too large for memory, sort runs in memory, then merge on disk.

**External Sort-Merge Algorithm:**
- **Pass 0 (Create Runs):** Read `M` blocks at a time (where `M` is buffer size), sort each chunk in memory, write as sorted run.
- **Merge Passes:** Merge runs pairwise using available buffers. If number of runs < M, merge all at once. Otherwise, merge in multiple passes.

**Cost:** `2 * b_r * (⌈log_{M-1}(⌈b_r / M⌉)⌉ + 1)` block transfers (approximately).

### 56.6. Join Algorithms

#### 56.6.1. Nested-Loop Join

For each tuple in outer relation `r`, scan entire inner relation `s`, and output matching pairs.

**Cost:** `n_r * b_s + b_r` block transfers (if `s` is inner and not all fits in memory). This is very expensive.

**Example:**
- `student`: 5000 tuples, 100 blocks
- `takes`: 10000 tuples, 400 blocks

If `student` is outer: Cost = 100 + 5000*400 = 2,000,100 block transfers.
If `takes` is outer: Cost = 400 + 10000*100 = 1,000,400 block transfers.

Smaller relation as outer reduces cost.

#### 56.6.2. Block Nested-Loop Join

For each block of outer relation, scan inner relation block by block.

**Cost:** `b_r * b_s + b_r` block transfers (if `s` is inner).

Using the same numbers:
- If `takes` is outer: Cost = 400 * 100 + 400 = 40,400 block transfers (much better).

**Optimization:** Use `M - 2` buffers for outer blocks; cost reduces further.

#### 56.6.3. Index Nested-Loop Join

If an index exists on the inner relation's join attribute, use it to look up matching tuples.

**Cost:** `b_r + n_r * c` where `c` is the cost of index lookup per tuple.

Using a B+ tree index on `takes.ID`:
- Cost ≈ 100 + 5000 * (height + 1) ≈ 25,000 or less. Dramatically lower.

### 56.7. Other Operations

**Duplicate Elimination:**
- Sort the data; duplicates become adjacent, remove them. Can be done during sorting merge.
- Or use hashing: tuples in same bucket are checked for duplicates.

**Projection:**
- Perform projection on each tuple, then eliminate duplicates (same as above).

**Aggregation:**
- Group tuples by aggregation attributes, compute aggregate.
- Can be done on the fly during sorting or hashing: for `count`, `sum`, `min`, `max`, update aggregate as tuples are encountered. For `average`, maintain `sum` and `count` separately.

---

## Module 57: Query Processing and Optimization – Part 2: Equivalence Rules and Plan Generation

### 57.1. Transformation of Relational Expressions

The optimizer generates equivalent expressions using **equivalence rules**. Two relational algebra expressions are **equivalent** if they produce the same set (or multiset in SQL) of tuples for all legal database instances.

**Key equivalence rules:**

#### Rule 1: Conjunctive Selection
```
σ_{θ1 ∧ θ2}(E) ≡ σ_{θ1}(σ_{θ2}(E))
```

#### Rule 2: Selection Commutative
```
σ_{θ1}(σ_{θ2}(E)) ≡ σ_{θ2}(σ_{θ1}(E))
```

#### Rule 3: Projection Cascading
```
π_{L1}(π_{L2}(E)) ≡ π_{L1}(E)   if L1 ⊆ L2
```
Only the last projection matters.

#### Rule 4: Selection and Cartesian Product / Theta Join
```
σ_θ(E1 × E2) ≡ E1 ⋈θ E2
σ_{θ1}(E1 ⋈θ2 E2) ≡ E1 ⋈_{θ1∧θ2} E2
```

#### Rule 5: Theta Join Commutative
```
E1 ⋈θ E2 ≡ E2 ⋈θ E1
```

#### Rule 6: Natural Join Associative
```
(E1 ⋈ E2) ⋈ E3 ≡ E1 ⋈ (E2 ⋈ E3)
```
Theta join associativity holds if conditions are properly distributed.

#### Rule 7: Selection Distributive over Join
```
σ_{θ1}(E1 ⋈θ2 E2) ≡ (σ_{θ1}(E1)) ⋈θ2 E2
  if θ1 involves only attributes of E1.

σ_{θ1∧θ2}(E1 ⋈θ3 E2) ≡ (σ_{θ1}(E1)) ⋈θ3 (σ_{θ2}(E2))
  if θ1 involves only E1, θ2 involves only E2.
```

#### Rule 8: Projection Distributive over Join
```
π_{L1∪L2}(E1 ⋈θ E2) ≡ π_{L1}(E1) ⋈θ π_{L2}(E2)
  if θ involves only attributes in L1∪L2.
```
If join attributes are not in projection lists, they must be added and later projected away.

#### Rule 9: Set Operations
- Union, intersection are commutative and associative; difference is not.
- Selection distributes over union, intersection, difference.
- Projection distributes over union.

### 57.2. Example: Optimizing a Query

**Query:**
```
Find names of instructors in Music department who taught a course in 2009, along with course titles.
```

**Original expression:**
```
π_{name,title}(σ_{dept_name='Music' ∧ year=2009}(instructor ⋈ teaches ⋈ course))
```

**Step 1: Push selection down using Rule 7:**
```
π_{name,title}(σ_{dept_name='Music'}(instructor) ⋈ σ_{year=2009}(teaches) ⋈ course)
```

**Step 2: Push projection early:**
```
π_{name,title}(π_{ID,name}(σ_{dept_name='Music'}(instructor)) ⋈ π_{ID,course_id}(σ_{year=2009}(teaches)) ⋈ π_{course_id,title}(course))
```

Now each relation is reduced before joining, minimizing intermediate sizes.

### 57.3. Evaluation Plan Generation

An **evaluation plan** is an annotated relational algebra tree where each operation has a chosen algorithm and index.

**Example annotations:**
- `σ_{dept_name='Music'}`: use index on `dept_name` (index 1).
- `σ_{year=2009}`: linear scan (no index).
- Join 1: merge join or nested-loop with index.
- Join 2: hash join.
- Final projection: sort to remove duplicates.

**Cost difference between plans can be enormous** (seconds vs days). So cost-based optimization is critical.

### 57.4. Cost-Based Optimization

**Steps:**
1. Generate equivalent expressions using equivalence rules.
2. Annotate each with possible algorithms (using available indexes).
3. Estimate cost of each plan using:
   - Statistical information: number of tuples, blocks, attribute distributions.
   - Cost formulas for algorithms (as in Module 56).
4. Choose the cheapest plan.

**Statistical information:**
- Number of tuples (`n_r`) and blocks (`b_r`) for each relation.
- Number of distinct values for attributes.
- Histograms or distribution of values.
- Index heights.

### 57.5. Heuristics and Dynamic Programming

**Pushing selections and projections early** reduces intermediate sizes.

**Join ordering:** Choose order that minimizes intermediate relation sizes. Associativity rules help explore orders.

**Dynamic Programming (System R style):**
- Generate plans for sub-expressions in increasing size.
- Maintain best plan for each sub-expression.
- Prune suboptimal plans using cost estimates.
- Avoid full exponential enumeration.

**Space optimization:** Share duplicate sub-expressions using pointers.

**Time optimization:** Use dynamic programming to avoid generating all equivalent expressions.

---

## Module 58: RDBMS Performance and Architecture

### 58.1. Performance Metrics

Three primary performance metrics:

1. **Throughput (TPS):** Transactions per second.
2. **Response Time:** Delay from transaction submission to result return.
3. **Availability:** Mean time to failure (MTTF); want high.

**Transactional level:**
- Concurrency control (locking, serializability)
- Query optimization
- Logging and recovery (log writes, checkpoints)

**System level:**
- System architecture (CPU, memory, disk, network)
- Database architecture (process management, buffer pool, lock manager)

### 58.2. Performance Tuning

**Hardware tuning:**
- Add more disks for higher I/O parallelism.
- Add more memory for larger buffer pool (fewer disk accesses).
- Upgrade to faster CPU.

**Database parameter tuning:**
- Buffer pool size.
- Checkpoint intervals (log size).
- Lock manager settings.

**Schema and index tuning:**
- Modify schema (e.g., denormalize for performance).
- Create/adjust indexes based on query patterns.
- Rewrite transactions for efficiency.

### 58.3. Scalability

Scalability is the ability to handle increasing amounts of data without sacrificing performance.

**Scaling dimensions:**
- **Volume**: More data.
- **Users**: More concurrent users.
- **Diversity of services**: More types of transactions.
- **Geographic expanse**: Distributed across locations.

### 58.4. RDBMS Architectures

#### 58.4.1. Centralized System

- Single computer, single user or few users.
- All components (CPU, memory, disk, I/O) connected by a common bus.
- Suitable for small applications.

#### 58.4.2. Client-Server System

- Clients request services; servers provide them.
- Network separates clients and servers.
- Frontend: forms, reports, query tools.
- Backend: SQL engine, transaction processing.
- Interface: ODBC, JDBC, etc.

**Server Types:**
- **Transaction Server**: Executes transactions, ships results. Common in RDBMS.
- **Data Server**: Ships data to clients for processing. Used in object-oriented systems.

**Server architecture:**
- User processes on client side.
- Server processes handle requests.
- Shared memory: buffer pool, log buffer, query plan cache.
- Background processes: lock manager, checkpoint, log writer.

#### 58.4.3. Parallel Systems

- Multiple processors and disks connected by fast network.
- Coarse-grained: few powerful processors.
- Massively parallel: thousands of small processors.

**Speedup:** Running a fixed-size problem on a larger system, ratio of times. Max speedup = n (linear).
**Scaleup:** Running a proportionally larger problem on a proportionally larger system, ratio of times. Max scaleup = 1 (linear).

**Why sub-linear speedup/scaleup?**
- Startup costs of parallel processes.
- Interference (shared bus, locks, data).
- Skew in task execution times.

**Interconnect architectures:**
- Bus: simple, limited.
- Mesh: root n by root n, diameter 2√n.
- Hypercube: log n diameter, most expensive.

**Parallel database architectures:**
- Shared memory
- Shared disk
- Shared nothing
- Hybrid

#### 58.4.4. Distributed Systems

- Data spread across multiple sites (nodes) connected by slower network.
- Homogeneous: same software/schema everywhere.
- Heterogeneous: different software/schema per site.
- Supports local and global transactions.

**Advantages:**
- Data sharing across sites.
- Autonomy.
- Higher availability via redundancy.

**Disadvantages:**
- More complex development.
- More potential for bugs.
- Increased processing overhead.

### 58.5. Scaling Up vs Scaling Out

**Vertical Scaling (Scale Up):**
- Add more resources to a single machine (CPU, RAM, disk).
- Cost-effective for moderate growth.
- Simpler maintenance.
- But single point of failure, downtime during upgrade.

**Horizontal Scaling (Scale Out):**
- Add more machines (nodes).
- Easier scaling, more resilience, better fault tolerance, higher performance.
- But more complex, higher initial cost.

**Approaches:**
- **Master-Slave Replication**: One master handles writes, multiple slaves handle reads.
- **Sharding**: Distribute data across multiple independent nodes by key ranges.

---

## Module 59: Non-Relational DBMS: NoSQL

### 59.1. Big Data

**Big data** refers to datasets so large and complex that traditional data processing applications are inadequate.

**5 V's of Big Data:**
1. **Volume**: Huge amounts of data.
2. **Variety**: Different data types (text, image, video, etc.).
3. **Velocity**: High speed of data generation.
4. **Variability**: Inconsistent data.
5. **Veracity**: Variable data quality.

**Examples:** Facebook, Google, Amazon, Twitter, etc., handle massive data.

### 59.2. What is NoSQL?

**NoSQL = Not Only SQL.**

A mechanism for storage and retrieval of data modeled in means other than tabular relations. It is a broad class of databases that depart from the relational model.

**History:**
- Hierarchical and network models predate relational.
- NoSQL term reintroduced in early 2000s.
- Inspired by Google's BigTable, Amazon's Dynamo, and the CAP theorem.

### 59.3. CAP Theorem

The **CAP Theorem** states that a distributed system can simultaneously guarantee at most two of the following three:

- **Consistency (C):** All nodes see the same data at the same time.
- **Availability (A):** Every request receives a response (success or failure).
- **Partition Tolerance (P):** System continues to operate despite network partitions.

**Since large systems must tolerate partitions, the choice is between C and A.**

- **RDBMS (ACID):** Prioritize Consistency and Availability (CA).
- **Many NoSQL (BASE):** Prioritize Availability and Partition tolerance (AP) or Consistency and Partition tolerance (CP).

**BASE:**
- **Basically Available**: System remains available.
- **Soft state**: State may change over time, even without input.
- **Eventual consistency**: System will become consistent eventually.

### 59.4. Eventual Consistency and Gossip Protocol

**Problem:** Multiple nodes hold copies of the same data. Writes happen at one node; other nodes may not see them immediately.

**Gossip protocol:** Nodes periodically exchange updates with a random subset of peers. Over time, updates propagate to all nodes.

**Eventual consistency:** If no new updates occur, eventually all nodes converge to the same value.

**Trade-off:** Instead of strong consistency (ACID), we accept eventual consistency (BASE) for better availability and partition tolerance.

### 59.5. Four Main Types of NoSQL Databases

#### 59.5.1. Key-Value Store

- Simple model: map of keys to values.
- Extremely fast and scalable.
- Examples: Amazon DynamoDB, Redis, Riak.

**API:**
- `get(key)`: Retrieve value.
- `put(key, value)`: Store value.
- `delete(key)`: Remove value.

**Advantages:** Simple, fast, fault tolerant, eventually consistent.
**Disadvantages:** Cannot model complex objects; no schema.

#### 59.5.2. Document Store

- Store semi-structured documents (XML, JSON).
- Documents can have nested fields, arrays.
- Examples: MongoDB, Couchbase, CouchDB.

**Example JSON document:**
```json
{
  "name": "John Smith",
  "addresses": [
    {"type": "home", "city": "Chennai"},
    {"type": "work", "city": "Mumbai"}
  ],
  "phone": ["9876543210", "9123456789"]
}
```

**Advantages:** Flexible schema, complex objects, hierarchical data.
**Disadvantages:** No declarative query language as powerful as SQL.

#### 59.5.3. Column-Family Store (Wide-Column Store)

- Inspired by Google's BigTable.
- Data stored in column families, each of which can have variable columns.
- Indexed by row key, column key, and timestamp.
- Examples: Cassandra, HBase, BigTable.

**Structure:**
- Row key uniquely identifies a row.
- Column families group related columns.
- Super columns group column families.
- Timestamp ensures eventual consistency.

**Advantages:** Scalable, sparse data, flexible columns.
**Disadvantages:** Less normalized, complex queries limited.

#### 59.5.4. Graph Store

- Data represented as nodes (entities) and edges (relationships) with properties.
- Ideal for highly connected data.
- Examples: Neo4j, OrientDB, ArangoDB.

**Query languages:** Cypher (Neo4j), Gremlin, SPARQL.

**Advantages:** Efficient traversal of relationships, path queries.
**Disadvantages:** Not well-suited for tabular or simple key-value data.

### 59.6. Relational vs Non-Relational

| Aspect | Relational (RDBMS) | Non-Relational (NoSQL) |
|---|---|---|
| Data model | Tables with rows/columns | Key-value, document, column, graph |
| Schema | Fixed, predefined | Flexible, dynamic |
| Consistency | Strong (ACID) | Eventual (BASE) |
| Scalability | Vertical, some horizontal | Horizontal (scale out) |
| Query language | SQL (declarative) | Proprietary, programmatic |
| Use cases | Transactions, enterprise | Web 2.0, big data |

**NoSQL is not a replacement for RDBMS**, but a complement for different needs.

---

## Module 60: Widely Used DBMSs and Course Summarization

### 60.1. Widely Used Relational DBMSs

The relational model organizes data into tables with rows, columns, and unique keys. SQL is the standard language for querying and maintaining RDBMSs.

**Key properties driving RDBMS dominance:**
- Simplicity
- Robustness
- Flexibility
- Performance
- Scalability (to a point)
- Compatibility in managing generic data

### 60.2. Proprietary RDBMSs

| DBMS | Vendor | Market Share | Primary Applications |
|---|---|---|---|
| Oracle | Oracle | ~48.8% | OLTP, data warehousing, mixed |
| Db2 | IBM | ~20% | Enterprise applications |
| SQL Server | Microsoft | ~17% | Business applications |
| Sybase (SAP) | SAP | ~4.7% | SAP applications |
| Teradata | Teradata | smaller | Data warehousing |

**Oracle:**
- First commercial release 1977.
- Latest: Oracle Database 19c.
- Sector-specific modules (CRM, ERP, etc.).
- Supports ODBC, JDBC, .NET, Python.

**IBM Db2:**
- Enterprise-grade database.
- Strong in mainframe environments.

**Microsoft SQL Server:**
- Integrated with Microsoft ecosystem.
- T-SQL language.

**Sybase (SAP):**
- Acquired by SAP; backbone of SAP applications.

**Teradata:**
- Specialized for data warehousing and analytics.

### 60.3. Free/Open Source RDBMSs

| DBMS | License | Latest Version (Aug 2021) | Applications |
|---|---|---|---|
| PostgreSQL | PostgreSQL License | 13.x | OLTP, data warehousing |
| MySQL | GPL/Oracle | 8.x | Web applications |
| SQLite | Public domain | 3.x | Embedded, mobile |

**PostgreSQL:**
- Advanced features, ACID compliant, extensible.
- Used in this course.

**MySQL:**
- Owned by Oracle, but open source.
- Most popular for web applications.

**SQLite:**
- Embedded, zero-configuration.
- Used in mobile apps, browsers, IoT.

### 60.4. Market Trends

**DB-Engines Ranking (August 2021):**
- Top 5: Oracle, MySQL, Microsoft SQL Server, PostgreSQL, IBM Db2.
- Next: SQLite, Microsoft Access, Teradata, SAP HANA, SAP Adaptive Server.

**Trend from 2013-2021:**
- Oracle and MySQL maintain top positions.
- PostgreSQL and MongoDB growing.
- Snowflake showing steep growth.

**All databases (including NoSQL):**
- Top 4 are relational: Oracle, MySQL, SQL Server, PostgreSQL.
- MongoDB, Redis are top NoSQL.
- Hybrid picture: relational still dominant, but NoSQL growing for web-scale applications.

### 60.5. Comparison Parameters

Various parameters can be compared across DBMSs:

- **Operating system support**: Linux, Windows, Mac, Android, etc.
- **Basic features**: ACID, triggers, stored procedures, etc.
- **Limits**: Maximum table size, columns, rows, database size.
- **Tables and views**: Support for types, temporary tables, materialized views.
- **Indexes**: B-tree, bitmap, hash, full-text.
- **Partitioning**: Range, hash, list, composite.
- **Access control**: Authentication, authorization, roles, privileges.

### 60.6. Course Recap

**Week 1: Introduction**
- Course overview, why DBMS over file systems, database design and engine basics.

**Week 2: Relational Model and SQL**
- Attributes, schema, instance, keys.
- Relational algebra, SQL DDL/DML, basic queries.

**Week 3: SQL Examples and Intermediate SQL**
- Nested subqueries, set operations, views, transactions, integrity constraints, authorization.
- Data modification (INSERT, UPDATE, DELETE).
- Advanced SQL: functions, procedures, triggers.

**Week 4: Formal Query Languages and ER Model**
- Relational algebra, tuple/domain calculus.
- Equivalence, predicate logic.
- ER modeling, ER diagrams, translation to relational schema.

**Week 5: Relational Database Design**
- Good design, redundancy, anomalies.
- Functional dependencies, Armstrong's axioms.
- Closure, canonical cover.
- BCNF, 3NF, decomposition algorithms.
- Lossless join, dependency preservation.

**Week 6: Further Normalization and Case Study**
- 2NF, 3NF, BCNF.
- Library Information System (LIS) case study.
- Multivalued dependencies, 4NF.
- Temporal data.

**Week 7: Application Development**
- Three-tier architecture.
- Web fundamentals, HTTP, HTML, scripting, servlets, JSP.
- SQL and native languages (ODBC, JDBC, embedded SQL).
- Python and PostgreSQL, Flask.
- RAD, performance, security, mobile apps.

**Week 8: Storage and File Structure**
- Algorithms and complexity.
- Linear data structures, arrays, linked lists, stacks, queues.
- Non-linear: trees, BST, 2-3-4 trees.
- Physical storage: disks, flash, tape, RAID.
- File organization, records, data dictionary, buffer management.

**Week 9: Indexing and Hashing**
- Ordered indices, primary/secondary, dense/sparse.
- B+ trees, B-trees.
- Static and dynamic hashing.
- Bitmap indices.
- Index design guidelines.

**Week 10: Transaction Management**
- Transaction concepts, states.
- Concurrency control, serializability.
- Lock-based protocols, two-phase locking.
- Deadlock handling.
- Time-based protocols.

**Week 11: Backup and Recovery**
- Backup strategies: full, incremental, differential.
- Cold vs hot backup.
- Transaction logs, log-based recovery.
- Checkpoints.
- RAID levels.

**Week 12: Query Optimization, Performance, NoSQL**
- Query processing pipeline.
- Selection, sorting, join algorithms.
- Equivalence rules, evaluation plans.
- RDBMS performance and scalability.
- NoSQL, big data, CAP theorem.
- Widely used DBMSs.

### 60.7. Final Guidance

- Read the textbook thoroughly.
- Practice SQL coding extensively.
- Practice database design and normalization.
- Strengthen algorithms and data structures.
- Study discrete mathematics.
- For NoSQL, study UML for broader modeling.

---

## Conclusion

Week 12 completes the course by covering the optimization of queries, the scalability of database architectures, the emergence of NoSQL for big data, and a comprehensive overview of widely used DBMSs. The journey has taken us from conceptual modeling to physical storage, from SQL to transactions, from backup to performance tuning. This knowledge provides a solid foundation for careers in database engineering, software development, data science, and beyond.